# Experimento — MLP, fase 3: treinar do zero e procurar padrões

A rede é treinada de novo, sem herdar arquitetura. As features de partida continuam sendo as do campeão da fase 2. A pergunta não é só "qual variante ganhou", mas **o que os resultados dizem sobre o modelo e os dados**.

1. **Grade:** todas as permutações de BN e Dropout por camada, com 1 a 4 camadas (120 planos), 3 larguras e ReLU, GELU e Leaky ReLU: 1.080 configurações em 10 divisões cada. BN e Dropout nunca ficam na mesma camada. O efeito médio de cada escolha sai de centenas de redes, o que vence o ruído que barrou as comparações isoladas.
2. **Estudo dos logits:** quando, em qual camada e em quais clientes a rede decora.
3. **Regularização:** começa pela camada apontada no estudo; depois vêm logit, intensidade, treino e balanceamento.
4. **Optuna:** refina em volta do que os passos anteriores mostraram.
5. **Features:** decide com 20 divisões novas o que ficou no limite da régua nas fases anteriores.
6. **Avaliação final e erros no teste.**

## As duas linhagens

| Linhagem | Escolhe por | Para que serve |
|---|---|---|
| `val` | KS de validação | O KS de teste continua comparável com os outros modelos do grupo |
| `teste` | KS de teste | Busca sem freio pelo melhor número; é otimista e fica fora da comparação entre modelos |

O teste nunca entra no ajuste: scaler, oversampling, pesos e parada usam só o treino. `otimismo_teste` mede quanto da escolha pelo teste vale fora daquele conjunto: escolhe em metade do teste e mede na outra metade.

## Como a GPU entra

Uma rede pequena sozinha é mais lenta na GPU do que na CPU. Por isso a grade, a repescagem e os blocos treinam em `pilha.py`: centenas de redes de uma vez, com os pesos de todas em um único tensor. Cada rede tem os próprios dados, sementes, plano e parada. Três etapas continuam em `modelo.py`, na CPU: o Optuna, porque cada trial tem um formato e pilhas pequenas perdem da CPU; o estudo dos logits, que olha dentro de cada rede; e a avaliação final, que é o número comparável com as fases anteriores. Antes da grade, a **conferência** mede as mesmas configurações nos dois treinadores e interrompe a execução se a pilha medir diferente.

## Como executar no Kaggle

1. *File → Import Notebook* → `experimentos/mlp/mlp_fase3.ipynb`, com o dataset anexado.
2. *Add-ons → Secrets:* `GITHUB_TOKEN` (com **Contents: Read and write**) e `WANDB_API_KEY`. **Nunca escreva os tokens neste notebook.**
3. *Settings:* Internet **On** e acelerador **GPU T4 x2**.
4. Rode com `MODO_TESTE = True`. Depois, com `False`, use *Save Version → Save & Run All (Commit)*.

# 0. Preparação do ambiente

In [ ]:
# ===== Configuração da execução (edite aqui) =====
REPO_URL = "https://github.com/diegoflyra/churn_project.git"
BRANCH = "diego/mlp-diagnostico"             # branch com o código
RESULTS_REPO_URL = REPO_URL      # onde espelhar os resultados ("" = não enviar ao GitHub)
RESULTS_BRANCH = "resultados"    # branch órfã só de resultados
RUN_NAME = ""                    # vazio = "mlp-fase3-AAAAMMDD-HHMM"; fixe um nome para retomar
RESUME_FROM = ""                 # "" | "github" | pasta com os resultados de uma execução anterior
CSV_PATH = "/kaggle/input/datasets/diegolyra/churn-ibm/telco_ibm.csv"                    # caminho do CSV; vazio = procurar no repositório e em /kaggle/input
FASE_ANTERIOR = "resultados/mlp-fase2-20261104-0505"   # de onde vêm as features de partida e a referência
USAR_GPU = True                  # treina em pilha na(s) GPU(s); sem GPU, cai para uma rede por vez em vários processos
DISPOSITIVO = "cpu"              # onde roda modelo.py (conferência, Optuna, logits e final): uma rede pequena por vez rende mais na CPU
N_PROCESSOS = 0                  # 0 = todos os núcleos
REPS_GRADE = 10                  # divisões por configuração na grade
TOP_REPESCAGEM = 10              # melhores de cada coluna medidas de novo em divisões que a grade não viu
REPS_REPESCAGEM = 10
REPS_BLOCO = 20                  # divisões por variante nos blocos
REPS_FEATURES = 20               # divisões nas decisões de features
REPS_CONFERENCIA = 6             # divisões por configuração na conferência entre a pilha e modelo.py
N_TRIALS = 300                   # trials do Optuna em cada linhagem
EPOCAS_LONGAS = 150              # épocas sem parada no estudo dos logits
N_SEMENTES = 5                   # sementes da avaliação final
USAR_WANDB = True                # False = não envia nada ao W&B, mesmo com a chave
USAR_GITHUB = True               # False = não espelha os resultados no GitHub, mesmo com o token
MODO_TESTE = False               # True = passada rápida para validar o notebook; os números não servem como resultado
HORAS_LIMITE = 10.5              # depois disso não começa etapas novas e encerra limpo (o Kaggle corta em 12 h)

In [ ]:
import importlib.util, os, shutil, subprocess, sys, time
from pathlib import Path

INICIO = time.time()


def _load_dotenv(path):
    """Tokens em um .env local (GITHUB_TOKEN, WANDB_API_KEY); nunca sobrescreve variáveis já definidas."""
    if os.path.isfile(path):
        for line in open(path, encoding="utf-8"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#") and value.strip() and not os.environ.get(key.strip()):
                os.environ[key.strip()] = value.strip().strip("'").strip('"')


def _secret(name):
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return ""


def _is_repo(p):
    return (p / "comum.py").exists() and (p / "experimentos" / "mlp" / "modelo.py").exists()


local = next((p for p in [Path.cwd(), *Path.cwd().parents] if _is_repo(p)), None)
if local:
    REPO_DIR = local
    _load_dotenv(REPO_DIR / ".env")
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    print(f"Código: cópia local em {REPO_DIR}")
else:
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    REPO_DIR = Path("/tmp/churn_project")
    url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1) if GITHUB_TOKEN else REPO_URL
    print("GitHub: usando GITHUB_TOKEN" if GITHUB_TOKEN else "GitHub: clone sem token")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, url, str(REPO_DIR)]).returncode != 0:
        raise RuntimeError(f"Falha ao clonar a branch {BRANCH}. Confira se ela foi publicada; repositório privado exige GITHUB_TOKEN.")
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])  # o token não fica salvo no clone
    print(f"Código: {REPO_DIR}")
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

# Instala só o que falta: o Kaggle já traz versões compatíveis com a GPU dele
IMPORTS = {"imbalanced-learn": "imblearn", "scikit-learn": "sklearn"}
faltam = [p.strip() for p in open("requirements.txt", encoding="utf-8")
          if p.strip() and importlib.util.find_spec(IMPORTS.get(p.strip(), p.strip())) is None]
if faltam:
    print("Instalando:", ", ".join(faltam))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltam])

In [ ]:
import json

sys.path[:0] = [str(REPO_DIR), str(REPO_DIR / "experimentos" / "mlp")]
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO_DIR), str(REPO_DIR / "experimentos" / "mlp"), os.environ.get("PYTHONPATH", "")])  # processos paralelos acham os módulos
import fluxo

WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content" if os.path.isdir("/content") else str(REPO_DIR)
RUN_NAME = RUN_NAME or time.strftime("mlp-fase3-%Y%m%d-%H%M")
if MODO_TESTE and not RUN_NAME.startswith("teste-"):
    RUN_NAME = "teste-" + RUN_NAME
# O teste nunca se mistura com a execução real, e rodar localmente não sobrescreve o que está versionado em resultados/
PASTA = os.path.join(WORKDIR, "execucoes", RUN_NAME) if WORKDIR == str(REPO_DIR) else os.path.join(WORKDIR, "resultados_" + RUN_NAME)
os.makedirs(PASTA, exist_ok=True)
os.environ["CHURN_RESULTADOS"] = PASTA
os.environ["CHURN_CSV"] = fluxo.achar_csv(CSV_PATH)
if DISPOSITIVO:
    os.environ["CHURN_DEVICE"] = DISPOSITIVO
N_PROCESSOS = N_PROCESSOS or os.cpu_count() or 1
GRADE = dict()
if MODO_TESTE:
    os.environ["CHURN_MAX_EPOCAS"] = "3"
    REPS_GRADE, TOP_REPESCAGEM, REPS_REPESCAGEM, REPS_BLOCO, REPS_FEATURES, N_TRIALS, EPOCAS_LONGAS, N_SEMENTES, REPS_CONFERENCIA = 2, 2, 2, 2, 2, 4, 3, 2, 2
    GRADE = dict(profundidades=(1, 2), ativacoes=("relu", "gelu"), larguras=(8,))
    print("MODO_TESTE: 3 épocas e grade reduzida. Os números não servem como resultado.")

if not USAR_GITHUB:
    RESULTS_REPO_URL = ""
if RESUME_FROM == "github":
    fluxo.baixar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)
elif RESUME_FROM:
    shutil.copytree(RESUME_FROM, PASTA, dirs_exist_ok=True)


def salvar():
    return fluxo.salvar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)


def salvar_json(nome, obj):
    json.dump(obj, open(os.path.join(PASTA, nome), "w"), indent=1, default=float)
    salvar()


_ultimo_backup = [time.time()]


def backup_periodico():
    if time.time() - _ultimo_backup[0] > 30 * 60:
        salvar(); _ultimo_backup[0] = time.time()


def continuar():
    return HORAS_LIMITE * 3600 - (time.time() - INICIO) > 0


import matplotlib.pyplot as plt, numpy as np, pandas as pd
from scipy.stats import ks_2samp
import comum, modelo as M, blocos as BL, diagnostico as D, fase3 as F3, logits as LG, pilha as P
import torch

GPUS = P.dispositivos_disponiveis() if USAR_GPU and torch.cuda.is_available() else None
F3.usar_pilha(GPUS)

pd.set_option("display.max_columns", None, "display.width", 220, "display.max_colwidth", 70, "display.max_rows", 80)
F3.abrir_cache(PASTA)
print(f"RUN_NAME = {RUN_NAME} | resultados em {PASTA}")
print(f"CSV: {os.environ['CHURN_CSV']} | pilha: {GPUS or 'desligada'} | modelo.py: {M.DEVICE} | processos: {N_PROCESSOS} | GitHub: {'sim' if GITHUB_TOKEN and RESULTS_REPO_URL else 'não'}")

wandb_run = None
wandb_key = _secret("WANDB_API_KEY") if USAR_WANDB else ""
if wandb_key or os.environ.get("WANDB_MODE") == "offline":
    import wandb
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
    wandb_run = wandb.init(project="churn-if702", entity=_secret("WANDB_ENTITY") or None, name=RUN_NAME, dir=WORKDIR,
                           group="mlp", config=dict(modelo="mlp", fase=3, processos=N_PROCESSOS, gpus=GPUS, teste=MODO_TESTE))
else:
    print("W&B desativado (sem chave ou USAR_WANDB = False).")


def logar(**k):
    if wandb_run is not None:
        wandb_run.log(k)


LIMITE_VIES = 0.005  # a conferência interrompe a execução acima disto
REFERENCIA = json.load(open(os.path.join(REPO_DIR, FASE_ANTERIOR, "blocos", "b12_arquitetura.json")))["campeao"]
print("Referência (campeão da fase 2):", json.dumps(REFERENCIA, ensure_ascii=False))

In [ ]:
# Verificações rápidas antes de gastar tempo de treino
folds, final, colunas = comum.carregar_dobras()
assert comum.assinatura_teste() == "e90ba6fab761f037", "o teste mudou: os números não seriam comparáveis com os outros modelos"
rng = np.random.default_rng(1); y_ = (rng.random(2000) < .265).astype(int); p_ = np.clip(rng.normal(.3 + .25 * y_, .2), 0, 1)
assert abs(comum.ks(y_, p_) - ks_2samp(p_[y_ == 1], p_[y_ == 0]).statistic) < 1e-9
# BN e Dropout nunca na mesma camada, em todos os planos da grade e em todo plano que o Optuna pode sortear
for pl in F3.planos():
    blocos_, atual = [], []
    for mod in M.MLP(5, len(pl), 8, "gelu", .2, pl).rede.children():
        if isinstance(mod, M.nn.Linear) and atual:
            blocos_.append(atual); atual = []
        atual.append(type(mod).__name__)
    assert all(not ("BatchNorm1d" in b and "Dropout" in b) for b in blocos_), pl
print(f"{len(F3.planos())} planos conferidos | teste com a assinatura oficial | KS conferido contra scipy")

In [ ]:
# A pilha mede o mesmo que modelo.py? O sorteio muda, então cada número muda; o que não pode aparecer é um viés.
if GPUS:
    _c = lambda **k: dict(hp={**REFERENCIA["hp"], "dropout": .2, **k}, dados=REFERENCIA["dados"], extras=REFERENCIA["extras"])
    CONFERIDAS = {"referencia": dict(hp={**REFERENCIA["hp"], "plano": "N"}, dados=REFERENCIA["dados"], extras=REFERENCIA["extras"]), "2x32_relu_BD": _c(camadas=2, unidades=32, plano="BD"),
                  "3x16_gelu_DDD": _c(camadas=3, unidades=16, ativ="gelu", plano="DDD"), "3x32_leaky_DBN": _c(camadas=3, unidades=32, ativ="leaky_relu", plano="DBN"), "4x64_gelu_BDBD": _c(camadas=4, unidades=64, ativ="gelu", plano="BDBD")}
    t0 = time.time()
    tab_conf, conf = F3.conferir_pilha(CONFERIDAS, range(1, 1 + REPS_CONFERENCIA), N_PROCESSOS)
    display(tab_conf.round(4)); display(pd.Series(conf).to_frame("valor").round(5))
    print(f"{(time.time() - t0) / 60:.1f} min")
    salvar_json("mlp_fase3_conferencia_pilha.json", dict(tabela=tab_conf.to_dict("records"), resumo=conf, dispositivos=GPUS, torch=torch.__version__))
    if not MODO_TESTE:
        for k in ("val", "teste"):
            assert abs(conf[f"vies_{k}"]) < LIMITE_VIES, f"a pilha mede {k} diferente de modelo.py: {conf}"

# 1. Grade: o efeito de cada escolha

Mede cada combinação de profundidade, largura, ativação e plano por camada (`N` nada, `B` BN, `D` Dropout). Com `lr`, `l2`, perda e paciência fixos no padrão, as diferenças vêm da estrutura.

**Como ler:** uma rede isolada não prova nada, mas o efeito médio sobre centenas delas sim. Um padrão só é confiável quando aponta para o mesmo lado na validação e no teste.

In [ ]:
CONFIGS = F3.grade(REFERENCIA, **GRADE)
CONFIGS["referencia_fase2"] = dict(hp={**REFERENCIA["hp"], "plano": "N"}, dados=REFERENCIA["dados"], extras=REFERENCIA["extras"])
print(len(CONFIGS), "configurações")
t0 = time.time()
tab, probs = F3.rodar_grade(PASTA, CONFIGS, range(1, REPS_GRADE + 1), N_PROCESSOS, continuar, backup_periodico)
print(f"{(time.time() - t0) / 60:.1f} min")
salvar()
display(tab.sort_values("val", ascending=False).head(12).round(4))
display(tab.sort_values("teste", ascending=False).head(12).round(4))
display(tab[tab.nome == "referencia_fase2"].round(4))

In [ ]:
# Os efeitos médios: a resposta mais confiável desta fase
grade_efeitos = tab[tab.nome != "referencia_fase2"]
pad = F3.padroes(grade_efeitos)
display(pad.round(4))
for fator in ("camadas", "unidades", "ativ"):
    display(grade_efeitos.groupby(fator)[["val", "teste", "epocas"]].mean().round(4))
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(grade_efeitos.val, grade_efeitos.teste, s=12, alpha=.6)
ref_ = tab[tab.nome == "referencia_fase2"]
ax.scatter(ref_.val, ref_.teste, c="k", marker="x", s=80, label="referência da fase 2")
ax.set_xlabel("KS de validação"); ax.set_ylabel("KS de teste"); ax.set_title(f"Grade: correlação {grade_efeitos.val.corr(grade_efeitos.teste):.2f}"); ax.legend(); plt.show()
salvar_json("mlp_fase3_efeitos.json", dict(padroes=pad.to_dict("records"), correlacao_val_teste=float(grade_efeitos.val.corr(grade_efeitos.teste))))

## Repescagem

Ser a melhor entre centenas infla o número. As melhores de cada coluna são medidas de novo em divisões que a grade não viu, e só então cada linhagem escolhe seu campeão.

In [ ]:
reps_r = range(11, 11 + REPS_REPESCAGEM)
top_val = list(tab.sort_values("val", ascending=False).nome.head(TOP_REPESCAGEM))
top_teste = list(tab.sort_values("teste", ascending=False).nome.head(TOP_REPESCAGEM))
rep_val = F3.repescagem(CONFIGS, top_val, reps_r, N_PROCESSOS).sort_values("val", ascending=False)
rep_teste = F3.repescagem(CONFIGS, top_teste, reps_r, N_PROCESSOS).sort_values("teste", ascending=False)
ref_rep = F3.repescagem(CONFIGS, ["referencia_fase2"], reps_r, N_PROCESSOS)
display(rep_val.round(4)); display(rep_teste.round(4)); display(ref_rep.round(4))
nome_val, nome_teste = rep_val.nome.iloc[0], rep_teste.nome.iloc[0]
camp = {"val": CONFIGS[nome_val], "teste": CONFIGS[nome_teste]}
print("Linhagem val:", nome_val, "| Linhagem teste:", nome_teste)

## Quanto a escolha pelo teste vale fora daquele conjunto?

Sorteia metades do teste, escolhe a melhor configuração na metade A e mede na metade B. É a checagem de que não estamos só acertando um conjunto específico.

In [ ]:
y_teste = F3._dados()[3].values.astype(int)
otimismo = F3.otimismo_teste(tab, probs, y_teste, nome_val, "referencia_fase2", n=20 if MODO_TESTE else 200)
display(pd.Series(otimismo).to_frame("valor"))
salvar_json("mlp_fase3_otimismo.json", otimismo)

### 📝 O que a grade mostra

- **Profundidade, largura e ativação importam?** _…_
- **BN e Dropout ajudam? Em qual camada?** _…_
- **Validação e teste concordam?** _…_
- **Escolher pelo teste rende fora daquele conjunto?** _…_

# 2. Estudo dos logits: onde a rede decora

Três perguntas sobre o campeão de cada linhagem:

- **Quando:** treinando sem parada, em que época a validação descola do treino? A confiança (|logit|) continua crescendo depois disso?
- **Em qual camada:** um classificador linear sobre a saída de cada camada mostra onde a lacuna entre treino e validação cresce. É ali que a regularização vai primeiro.
- **Em quais clientes:** comparando cada cliente quando está no treino e quando está fora, quais grupos a rede decora mais do que a média?

In [ ]:
def estudar_logits(c, rotulo):
    arq = os.path.join(PASTA, f"mlp_fase3_logits_{rotulo}.json")
    if os.path.isfile(arq):
        e = json.load(open(arq))
        print(f"{rotulo}: já concluído, reaproveitado.")
        return e
    e = LG.estudar(c, EPOCAS_LONGAS, n_jobs=N_PROCESSOS)
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    for pre in ("treino", "val", "teste"):
        ax[0].plot(e["curva"].epoca, e["curva"][f"{pre}_ce"], label=pre); ax[1].plot(e["curva"].epoca, e["curva"][f"{pre}_ks"], label=pre); ax[2].plot(e["curva"].epoca, e["curva"][f"{pre}_abs_logit"], label=pre)
    for a, t_ in zip(ax, ("Entropia cruzada", "KS", "|logit| médio")):
        a.axvline(e["marcos"]["epoca_min_ce_val"], c="gray", ls="--"); a.set_title(f"{rotulo}: {t_}"); a.set_xlabel("época"); a.legend()
    plt.show()
    display(pd.Series(e["marcos"]).to_frame("valor").round(4))
    display(e["camadas"].round(4))
    print("Camada crítica:", e["camada_critica"])
    display(e["por_rotulo"].round(3)); display(e["por_ambiguidade"].round(3))
    display(e["grupos_longa"].head(10).round(3)); display(e["grupos_longa"].tail(5).round(3))
    print(f"Grupos com q < 0,05: {int((e['grupos_longa'].q_bh < .05).sum())} de {len(e['grupos_longa'])}")
    j = LG.para_json(e)
    salvar_json(os.path.basename(arq), j)
    return j


logits_ini = {}
for lin in ("val", "teste"):
    if continuar():
        logits_ini[lin] = estudar_logits(camp[lin], f"inicio_{lin}")

### 📝 O que os logits mostram

- **Em que época a rede começa a decorar?** _…_
- **Em qual camada a lacuna cresce?** _…_
- **Quais clientes são decorados? Os ambíguos, alguma classe, algum grupo?** _…_

# 3. Regularização guiada pelo estudo

Cada bloco parte do campeão anterior da linhagem. A regra é a mesma de antes, aplicada na coluna da linhagem: ganho médio acima da régua de ruído e vitória em pelo menos 80% das divisões. As duas colunas aparecem na tabela, então dá para ver se validação e teste concordam.

In [ ]:
reps_b = range(21, 21 + REPS_BLOCO)
ETAPAS = [("b13_onde_regularizar", "Regularizar a camada apontada pelos logits supera as outras posições?", lambda c, lin: F3.variantes_estrutura(c, logits_ini[lin]["camada_critica"] if lin in logits_ini else None)),
          ("b14_logit", "Atacar a confiança direto no logit ajuda?", lambda c, lin: F3.variantes_logit()),
          ("b15_intensidade", "Mais L2 ou outra taxa de Dropout ajudam?", lambda c, lin: F3.variantes_intensidade(c)),
          ("b16_treino", "Outro lote, lr, otimizador, paciência ou critério de parada ajudam?", lambda c, lin: F3.variantes_treino()),
          ("b17_balanceamento", "Trocar as cópias do oversampling por peso na classe positiva ajuda?", lambda c, lin: F3.variantes_balanceamento())]
for nome, pergunta, gera in ETAPAS:
    for lin in ("val", "teste"):
        if continuar():
            camp[lin], t = F3.bloco(PASTA, nome, pergunta, gera(camp[lin], lin), camp[lin], lin, reps_b, n_jobs=N_PROCESSOS)
            display(t.round(4))
            logar(bloco=nome, linhagem=lin, val=float(t.val.max()), teste=float(t.teste.max()))
            backup_periodico()
salvar()
print(json.dumps(camp, ensure_ascii=False, indent=1))

# 4. Optuna: refinar em volta do que apareceu

O primeiro trial é o campeão da linhagem. O espaço inclui plano por camada, funil, ativação, regularizações e critério de parada. A linhagem `val` maximiza a validação e a linhagem `teste` maximiza o teste. O resultado só substitui o campeão se passar pela mesma regra, em divisões novas.

In [ ]:
estudos = {}
for lin in ("val", "teste"):
    if continuar():
        st, melhor = F3.estudar(PASTA, lin, camp[lin], N_TRIALS, n_jobs=N_PROCESSOS, continuar=continuar, ao_salvar=backup_periodico)
        estudos[lin] = st
        feitos = [t for t in st.trials if t.state.name == "COMPLETE"]
        print(f"{lin}: {len(feitos)} trials | melhor {st.best_value:.4f} | {st.best_params}")
        q = pd.DataFrame([dict(val=t.user_attrs["val"], teste=t.user_attrs["teste"]) for t in feitos])
        print(f"Correlação entre validação e teste nos trials: {q.val.corr(q.teste):.2f}")
        if melhor["hp"] != camp[lin]["hp"]:
            camp[lin], t = F3.bloco(PASTA, "b18_optuna", "O melhor trial supera o campeão em divisões novas?", {"melhor trial": dict(hp=melhor["hp"])}, camp[lin], lin, range(41, 41 + REPS_BLOCO), n_jobs=N_PROCESSOS)
            display(t.round(4))
salvar()

# 5. Features com mais divisões

`tenure_ate_3`, `sem_gender` e `delta_preco` ficaram no limite da régua nas fases anteriores. Com a rede nova e 20 divisões, a média decide.

In [ ]:
for lin in ("val", "teste"):
    if continuar():
        camp[lin], t = F3.bloco(PASTA, "b19_features", "Com mais divisões, as features no limite da régua ajudam?", F3.variantes_features(camp[lin]), camp[lin], lin, range(61, 61 + REPS_FEATURES), n_jobs=N_PROCESSOS)
        display(t.round(4))
salvar()
print(json.dumps(camp, ensure_ascii=False, indent=1))

# 6. Avaliação final e erros no teste

Treino inteiro, 5 sementes, limiar vindo da validação. `val` é a linha comparável com os outros modelos. `teste` é o melhor número encontrado, otimista por construção.

In [ ]:
finais = {}
if continuar():
    for rot, c in (("referencia", REFERENCIA), ("val", camp["val"]), ("teste", camp["teste"])):
        finais[rot] = BL.final_campeao(c, N_SEMENTES)
        m = finais[rot]["teste"]
        print(f"{rot:<11} KS = {m['ks']:.4f} | IC 95% {np.round(m['ks_ic95'], 4)} | AUROC {m['auroc']:.4f} | por semente {np.round(finais[rot]['ks_teste_por_semente'], 4)}")
    y_f = F3._dados()[3].values.astype("float32")
    P = {k: np.array(v["probabilidades_teste"]) for k, v in finais.items()}
    comparacoes = {"val - referencia": F3.bootstrap_delta(y_f, P["val"], P["referencia"]), "teste - referencia": F3.bootstrap_delta(y_f, P["teste"], P["referencia"]), "teste - val": F3.bootstrap_delta(y_f, P["teste"], P["val"])}
    display(pd.DataFrame(comparacoes).T)
    salvar_json("mlp_fase3_final.json", dict(finais=finais, comparacoes=comparacoes, avisos=dict(val="escolhido só pela validação; comparável com os outros modelos", teste="escolhido pelo teste; otimista, fora da comparação entre modelos")))

In [ ]:
# Em quem o modelo erra no teste e o custo de trocar um erro pelo outro
erros_finais = {}
for lin in finais:
    if lin == "referencia":
        continue
    grupos, um, pares = F3.erros_no_teste(finais[lin]["probabilidades_teste"])
    troca = F3.trade_off_erros(finais[lin]["probabilidades_teste"], finais[lin]["teste"]["limiar"])
    print(f"Linhagem {lin}: resíduos com q < 0,05: {int((um.q_bh < .05).sum())} de {len(um)} grupos, {int((pares.q_bh < .05).sum())} de {len(pares)} pares")
    display(grupos.head(10).round(3)); display(pares.head(10).round(3)); display(troca.round(4))
    erros_finais[lin] = dict(por_grupo=grupos.to_dict("records"), residuos=um.to_dict("records"), pares=pares.head(100).to_dict("records"), trade_off=troca.to_dict("records"))
if erros_finais:
    salvar_json("mlp_fase3_erros_teste.json", erros_finais)

In [ ]:
# A regularização tirou a decoreba? E ainda sobra sinal para o modelo aproveitar?
if continuar():
    for lin in ("val", "teste"):
        estudar_logits(camp[lin], f"fim_{lin}")
if continuar():
    X_d, y_d, p_d = D.oof(camp["val"], min(N_SEMENTES, 3))
    sinal = D.sinal_restante(X_d, y_d, p_d); display(sinal.round(4))
    curva_d = D.curva_aprendizado(camp["val"], n_seeds=min(N_SEMENTES, 3)); display(curva_d.round(4))
    salvar_json("mlp_fase3_diagnostico.json", dict(sinal=sinal.to_dict("records"), curva=curva_d.to_dict("records")))

### 📝 Análise da fase 3

- **Que padrões apareceram nos dados e no modelo?** _…_
- **Qual regularização atuou onde a rede decorava?** _…_
- **Quanto rendeu em relação à fase 2, nas duas linhagens?** _…_
- **Próxima hipótese:** _…_

In [ ]:
salvar_json("mlp_fase3_campeoes.json", camp)
salvar()
if wandb_run is not None:
    wandb_run.summary.update(dict(campeao_val=json.dumps(camp["val"]), campeao_teste=json.dumps(camp["teste"])))
    wandb_run.finish()
print(f"Tempo total: {(time.time() - INICIO) / 3600:.2f} h")
print("Resultados em", PASTA)
print(sorted(os.listdir(PASTA)))